# Keep-alive and resource monitor

Run the one code cell and leave the notebook open until the batch is finished. Every 15 seconds it prints CPU cores in use, RAM used against the pod's limit, free disk, the number of pipeline processes and (every 10 minutes) whether the S3 credentials still work. The kernel stays active, so the pod is not shut down for inactivity; an error inside the loop is printed and the loop goes on. It reads system files only and holds almost no memory. Stop it with Kernel > Interrupt.

In [ ]:
# Keep-alive monitor. Why (user, 7 Oct 2026): the pod shuts down after a while without kernel activity, which stopped
# long batch runs twice. This loop keeps the kernel busy and prints the pod's load every 15 seconds. It is tiny (no extra
# packages, reads /proc and the cgroup files only), so it cannot cause an out-of-memory stop itself.
# Stop it with the notebook's stop button (Kernel > Interrupt).
import os, time, subprocess
from datetime import datetime

def _read(p, default=None):
    try:
        return open(p).read().strip()
    except OSError:
        return default

def mem_used_limit():
    used = _read("/sys/fs/cgroup/memory.current")
    limit = _read("/sys/fs/cgroup/memory.max")
    if used and limit and limit != "max":
        return int(used), int(limit)
    info = dict(l.split(":", 1) for l in open("/proc/meminfo"))
    tot = int(info["MemTotal"].split()[0]) * 1024
    avail = int(info["MemAvailable"].split()[0]) * 1024
    return tot - avail, tot

def cpu_usage_usec():
    s = _read("/sys/fs/cgroup/cpu.stat", "")
    for line in s.splitlines():
        if line.startswith("usage_usec"):
            return int(line.split()[1])
    return None

def cpu_quota():
    q = _read("/sys/fs/cgroup/cpu.max", "max 100000").split()
    return os.cpu_count() if q[0] == "max" else int(q[0]) / int(q[1])

def pipeline_jobs():
    out = subprocess.run(["pgrep", "-af", "sar_pipeline"], capture_output=True, text=True).stdout.splitlines()
    return [l for l in out if "pgrep" not in l]

def s3_ok():
    """True / False for the storage credentials kept in secrets/aws_env.sh (None when there is no such file). Why
    (user, 7 Oct 2026): expired session tokens stop only the tree cutting; the user wants to see it at once."""
    env_file = os.path.join(os.path.dirname(os.path.abspath("")), "secrets", "aws_env.sh")
    if not os.path.exists(env_file):
        env_file = os.path.abspath(os.path.join("..", "secrets", "aws_env.sh"))
    if not os.path.exists(env_file):
        return None
    r = subprocess.run(["bash", "-c", f"source '{env_file}' && aws sts get-caller-identity >/dev/null 2>&1"])
    return r.returncode == 0

INTERVAL_S = 15            # one line every 15 seconds (user, 7 Oct 2026)
S3_EVERY = 40              # S3 check every 40 lines = every 10 minutes

print(f"{datetime.now():%H:%M:%S}  monitor started; one line every {INTERVAL_S} s; stop it with Kernel > Interrupt",
      flush=True)
prev, t_prev = cpu_usage_usec(), time.time()
time.sleep(2)
tick, s3 = 0, None
while True:
    try:                   # nothing inside may stop the loop: an error is printed and the next line comes as usual
        now, t_now = cpu_usage_usec(), time.time()
        cores = (now - prev) / 1e6 / (t_now - t_prev) if now is not None and prev is not None else float("nan")
        prev, t_prev = now, t_now
        used, limit = mem_used_limit()
        st = os.statvfs("/home/jovyan")
        free_gb = st.f_bavail * st.f_frsize / 1e9
        pct = 100 * used / limit
        warn = "  <-- RAM above 85 %" if pct > 85 else ""
        if tick % S3_EVERY == 0:
            s3 = s3_ok()
        warn += "  <-- S3 CREDENTIALS EXPIRED: paste fresh ones into secrets/aws_env.sh" if s3 is False else ""
        print(f"{datetime.now():%H:%M:%S}  CPU {cores:4.1f}/{cpu_quota():.0f} cores  RAM {used/1e9:5.1f}/{limit/1e9:.0f} GB "
              f"({pct:4.1f} %)  disk free {free_gb:5.0f} GB  pipeline processes {len(pipeline_jobs())}"
              f"  S3 {'ok' if s3 else ('none' if s3 is None else 'EXPIRED')}{warn}", flush=True)
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print(f"{datetime.now():%H:%M:%S}  monitor error (it goes on): {type(e).__name__}: {e}", flush=True)
    tick += 1
    time.sleep(INTERVAL_S)
